# Snippet from Math-SPD-Metrics-and-Riemannian-Geometry.md


In [ ]:
def analyze_essay_clusters(embeddings: np.ndarray, labels: np.ndarray, r: int = 4):
    """
    Cluster essays using learned SPD metric.
    
    Args:
        embeddings: Essay embeddings (n x D).
        labels: Ground truth labels (n,).
        r: Low-rank dimension.
    
    Returns:
        Dictionary with clustering metrics.
    """
    # Learn metric from data
    L = SymbolicManifoldMetric.low_rank_fit(embeddings, r=r)
    metric = SymbolicManifoldMetric(L, delta=1e-3)
    
    # Compute pairwise distances
    n = embeddings.shape[0]
    distances_M = np.zeros((n, n))
    distances_euclidean = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
            d_M = metric.distance(embeddings[i], embeddings[j])
            d_euc = np.linalg.norm(embeddings[i] - embeddings[j])
            distances_M[i, j] = distances_M[j, i] = d_M
            distances_euclidean[i, j] = distances_euclidean[j, i] = d_euc
    
    # Analyze within-class vs between-class distances
    within_class_M = []
    between_class_M = []
    within_class_euc = []
    between_class_euc = []
    
    for i in range(n):
        for j in range(i+1, n):
            if labels[i] == labels[j]:
                within_class_M.append(distances_M[i, j])
                within_class_euc.append(distances_euclidean[i, j])
            else:
                between_class_M.append(distances_M[i, j])
                between_class_euc.append(distances_euclidean[i, j])
    
    # Compute separation metrics
    separation_M = np.mean(between_class_M) / (np.mean(within_class_M) + 1e-10)
    separation_euc = np.mean(between_class_euc) / (np.mean(within_class_euc) + 1e-10)
    
    return {
        "separation_spd": separation_M,
        "separation_euclidean": separation_euc,
        "improvement": separation_M / separation_euc,
        "within_mean_spd": np.mean(within_class_M),
        "between_mean_spd": np.mean(between_class_M),
        "condition_number": metric.condition_number()
    }

# Example with synthetic data
np.random.seed(42)
n_samples = 50
D = 20
n_classes = 3

# Create synthetic essay embeddings with class structure
embeddings = []
labels = []
for c in range(n_classes):
    center = np.random.randn(D) * 3
    class_samples = center + np.random.randn(n_samples // n_classes, D) * 0.5
    embeddings.append(class_samples)
    labels.extend([c] * (n_samples // n_classes))

embeddings = np.vstack(embeddings)
labels = np.array(labels)

results = analyze_essay_clusters(embeddings, labels, r=4)

print("Essay Clustering Analysis")
print("=" * 50)
print(f"SPD Separation: {results['separation_spd']:.3f}")
print(f"Euclidean Separation: {results['separation_euclidean']:.3f}")
print(f"Improvement Factor: {results['improvement']:.3f}x")
print(f"Within-class mean (SPD): {results['within_mean_spd']:.3f}")
print(f"Between-class mean (SPD): {results['between_mean_spd']:.3f}")
print(f"Metric condition number: {results['condition_number']:.2e}")
